# AE3 (S30): Transformación y Enriquecimiento de Datos

## 1. Necesidad, pregunta y alcance

La transformación prepara WA_Crash para analizar: **¿cómo se distribuye la severidad de los accidentes registrados entre 2019 y 2023 según el tipo de calzada, la presencia de intersección, la franja horaria y los tipos de vehículos involucrados?**

| Dimensión | Variables de origen | Resultado esperado |
|---|---|---|
| Periodo | `CRASH_DATE` | Accidentes fechados entre 2019 y 2023 |
| Severidad | `SEVERITY` | Distribución de accidentes por categoría |
| Tipo de calzada | `CWAY` | Comparación descriptiva por categoría vial |
| Intersección | `INTERSECTION_NO`, `INTERSECTION_DESC` | Estado registrado; faltante no equivale a ausencia |
| Franja horaria | `CRASH_TIME` | Hora HHMM para la posterior derivación de franjas |
| Vehículos/personas involucrados | columnas `TOTAL_*_INVOLVED` | Conteos por tipo para el enriquecimiento posterior |

**Limitaciones:** no hay datos de tráfico ni de exposición; por tanto, no se puede afirmar que un tipo de calzada sea más riesgoso. Una intersección no registrada no equivale a que no exista. Las comparaciones son descriptivas y se limitan a accidentes registrados.

## 2. Carga del dataset y copia inalterada

Se carga el split `train` de Hugging Face y se conserva una copia independiente para comparar el resultado. El dataset crudo no se exporta al repositorio.

In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

RAIZ_REPO = Path.cwd().resolve()
for candidata in (RAIZ_REPO, *RAIZ_REPO.parents):
    if (candidata / "AE3" / "src" / "clases_po").is_dir():
        RAIZ_REPO = candidata
        break
else:
    raise FileNotFoundError("No se encontró la raíz del repositorio con AE3/src/clases_po.")

SOURCE_DIR = RAIZ_REPO / "AE3" / "src"
RESULTADOS_DIR = RAIZ_REPO / "AE3" / "resultados"
sys.path.insert(0, str(SOURCE_DIR))

from clases_po.hf_loader import HFDatasetLoader
from clases_po.preprocesamiento import PreprocesadorCrash
from clases_po.reportes import Reportes

RESULTADOS_DIR.mkdir(parents=True, exist_ok=True)
loader = HFDatasetLoader(dataset_name="kalley/WA_Crash", split="train")
df = loader.load()
df_original = df.copy(deep=True)
print(f"Dataset: {loader.info()}")
print(f"Columnas: {df_original.columns.tolist()}")

README.md:   0%|          | 0.00/1.98k [00:00<?, ?B/s]

Crash_Information_WA.csv: reconstructing file:   0%|          |  0.00B / 29.9MB            

Crash_Information_WA.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/127102 [00:00<?, ? examples/s]

Dataset: {'dataset': 'kalley/WA_Crash', 'config': None, 'split': 'train', 'filas': 127102, 'columnas': 26}
Columnas: ['X', 'Y', 'ACC_ID', 'ROAD_NO', 'ROAD_NAME', 'COMMON_ROAD_NAME', 'CWAY', 'SLK', 'INTERSECTION_NO', 'INTERSECTION_DESC', 'LONGITUDE', 'LATITUDE', 'CRASH_DATE', 'CRASH_TIME', 'ACCIDENT_TYPE', 'SEVERITY', 'EVENT_NATURE', 'EVENT_TYPE', 'TOTAL_BIKE_INVOLVED', 'TOTAL_TRUCK_INVOLVED', 'TOTAL_HEAVY_TRUCK_INVOLVED', 'TOTAL_MOTOR_CYCLE_INVOLVED', 'TOTAL_OTHER_VEHICLES_INVOLVED', 'TOTAL_PEDESTRIANS_INVOLVED', 'OBJECTID', 'GlobalID']


## 3. Transformación y validación

El diagnóstico permite revisar los tipos y los valores faltantes de origen antes de convertir columnas. No se altera `df_original`.

In [2]:
preprocesador = PreprocesadorCrash()
diagnostico_inicial = preprocesador.diagnosticar(df_original)
display(diagnostico_inicial)

,columna,tipo_actual,no_nulos,nulos,porcentaje_nulos,ejemplo
0,X,float64,127102,0,0.00,115.907761
1,Y,float64,127102,0,0.00,-31.966608
2,ACC_ID,int64,127102,0,0.00,10972646
3,ROAD_NO,object,127102,0,0.00,1290190
4,ROAD_NAME,object,127102,0,0.00,Cookham Rd
5,COMMON_ROAD_NAME,object,127102,0,0.00,Cookham Rd
6,CWAY,object,127102,0,0.00,S
7,SLK,float64,127102,0,0.00,0.76
8,INTERSECTION_NO,float64,68123,58979,46.40,13935.0
9,INTERSECTION_DESC,object,68123,58979,46.40,Melville Mandurah Hwy & Cockburn Rd


**Interpretación del diagnóstico inicial:** la tabla registra, para cada variable, el tipo recibido, la cantidad y porcentaje de nulos y un valor de ejemplo. Los nulos de hora e intersección se conservarán; no se imputan ni se interpretan como ausencia de intersección.

In [3]:
df_limpio = preprocesador.procesar(df_original)
print(f"DataFrame limpio: {df_limpio.shape[0]} filas x {df_limpio.shape[1]} columnas")
validacion_correcta = preprocesador.validar(df_limpio)
assert validacion_correcta, "Falló al menos un control de validación; revisar los resultados."

DataFrame limpio: 127102 filas x 18 columnas
✔ Fechas completas dentro del periodo 2019-2023
✔ ACC_ID único y no nulo
✔ Tipos esperados de fechas, horas y conteos
✔ Horas no nulas válidas en formato HHMM
✔ Categorías de SEVERITY esperadas


La siguiente tabla muestra cuántos registros y qué porcentaje representan las categorías observadas en `SEVERITY` y `CWAY` dentro del DataFrame limpio.

In [4]:
filas_frecuencia = []
for dimension in ("SEVERITY", "CWAY"):
    conteos = df_limpio[dimension].value_counts(dropna=False)
    porcentajes = df_limpio[dimension].value_counts(dropna=False, normalize=True) * 100
    for categoria, cantidad in conteos.items():
        filas_frecuencia.append(
            {
                "dimension": dimension,
                "categoria": "Sin dato" if pd.isna(categoria) else categoria,
                "accidentes": int(cantidad),
                "porcentaje": round(float(porcentajes.loc[categoria]), 2),
            }
        )

tabla_frecuencias = pd.DataFrame(filas_frecuencia)
display(tabla_frecuencias)

,dimension,categoria,accidentes,porcentaje
0,SEVERITY,PDO Major,68868,54.18
1,SEVERITY,PDO Minor,34750,27.34
2,SEVERITY,Medical,16485,12.97
3,SEVERITY,Hospital,6264,4.93
4,SEVERITY,Fatal,735,0.58
5,CWAY,S,69954,55.04
6,CWAY,L,44758,35.21
7,CWAY,R,12390,9.75


**Interpretación de frecuencias:** sobre 127.102 accidentes, `SEVERITY` se distribuye en PDO Major: 68.868 (54,18 %), PDO Minor: 34.750 (27,34 %), Medical: 16.485 (12,97 %), Hospital: 6.264 (4,93 %) y Fatal: 735 (0,58 %). Para `CWAY`, S representa 69.954 (55,04 %), L 44.758 (35,21 %) y R 12.390 (9,75 %). Son proporciones descriptivas del conjunto limpio.

El log siguiente documenta cada regla y sus conteos, incluidos los controles sin problemas. Los filtros de fechas y duplicados justifican las reducciones de filas; las horas inválidas y conteos imposibles se convierten en nulos sin eliminar la fila.

In [5]:
tabla_decisiones = preprocesador.tabla_log()
display(tabla_decisiones)

,paso,regla,criterio/justificación,registros_afectados,filas_antes,filas_despues
0,fechas no interpretables,Excluir registros sin fecha interpretable,Sin fecha no es posible asignar el accidente a...,0,127102,127102
1,filtro de periodo,Conservar accidentes entre 2019 y 2023 inclusive,La pregunta de análisis limita el alcance a es...,0,127102,127102
2,duplicados completos,Conservar una copia de cada fila idéntica,Las repeticiones exactas no representan regist...,0,127102,127102
3,duplicados por ACC_ID,Conservar el primer registro de cada identific...,ACC_ID identifica el accidente; los registros ...,0,127102,127102
4,consistencia de intersección,Revisar el patrón de faltantes de INTERSECTION...,58979 registros tienen ambas variables nulas; ...,0,127102,127102
5,horas inválidas,Invalidar horas fuera de HHMM (0000-2359 y min...,Las horas inválidas se convierten en nulos; no...,0,127102,127102
6,normalización de texto,Eliminar espacios sobrantes al inicio y al final,Se recortan únicamente los valores que contení...,0,127102,127102
7,categorías de SEVERITY,Revisar variantes por mayúsculas/minúsculas o ...,No se encontraron categorías duplicadas por ma...,0,127102,127102
8,categorías de CWAY,Revisar variantes por mayúsculas/minúsculas o ...,No se encontraron categorías duplicadas por ma...,0,127102,127102
9,valores de SEVERITY,"Contrastar categorías con FATAL, HOSPITAL, MED...",No se eliminan ni sustituyen categorías descon...,0,127102,127102


**Interpretación del log:** `registros_afectados` expresa las filas o valores involucrados en la regla; `filas_antes` y `filas_despues` permiten distinguir los filtros de las revisiones que solo modifican valores o columnas. No se rellenan los faltantes de intersección ni de hora.

In [6]:
comparacion = preprocesador.comparar_antes_despues(df_original, df_limpio)
display(comparacion)

,columna,filas_antes,columnas_antes,tipo_antes,nulos_antes,filas_despues,columnas_despues,tipo_despues,nulos_despues
0,X,127102,26,float64,0,127102,18,descartada,NaN
1,Y,127102,26,float64,0,127102,18,descartada,NaN
2,ACC_ID,127102,26,int64,0,127102,18,int64,0.0
3,ROAD_NO,127102,26,object,0,127102,18,descartada,NaN
4,ROAD_NAME,127102,26,object,0,127102,18,descartada,NaN
5,COMMON_ROAD_NAME,127102,26,object,0,127102,18,descartada,NaN
6,CWAY,127102,26,object,0,127102,18,string,0.0
7,SLK,127102,26,float64,0,127102,18,descartada,NaN
8,INTERSECTION_NO,127102,26,float64,58979,127102,18,float64,58979.0
9,INTERSECTION_DESC,127102,26,object,58979,127102,18,string,58979.0


**Interpretación de la comparación:** las columnas descartadas aparecen como `descartada`; para las conservadas se muestran los tipos y nulos en ambos estados. El resultado contiene solo las variables pertinentes y mantiene los nombres originales en mayúsculas.

## 4. Enriquecimiento de datos

El enriquecimiento usa solo columnas ya existentes en `df_limpio`; **no se combina ninguna otra tabla**, por lo que no hay uniones que puedan multiplicar registros. Toda la lógica está en la clase `Enriquecimiento` (`AE3/src/clases_po/enriquecimiento.py`).

**Sobre el modelo de NVIDIA:** `nvidia/nemotron-3.5-parse` es un modelo de extracción de documentos (recibe la imagen de una página y devuelve Markdown); no redacta conocimiento de negocio a partir de texto. Por eso las variables y las interpretaciones se calculan con reglas deterministas y el modelo se usa como **verificador independiente de lectura** de las tablas (sección 5.3), conectado mediante `ClienteNvidia` (`llm_nvidia.py`) con el secreto `API_KEY_NVIDIA`.

En la API alojada de NVIDIA el identificador nvidia/nemotron-3.5-parse no fue aceptado; respondió nvidia/nemotron-parse (versión anterior de la misma familia). La columna estado de validacion_llm_tablas.csv registra el modelo que respondió en cada ejecución.

In [7]:
from clases_po.enriquecimiento import Enriquecimiento
from clases_po.llm_nvidia import ClienteNvidia
from IPython.display import Markdown

enriquecimiento = Enriquecimiento()
df_enriquecido = enriquecimiento.agregar_variables(df_limpio)
assert len(df_enriquecido) == len(df_limpio), "El enriquecimiento no debe cambiar el número de filas."
print(f"DataFrame enriquecido: {df_enriquecido.shape[0]} filas x {df_enriquecido.shape[1]} columnas")
display(df_enriquecido[list(Enriquecimiento.NUEVAS_VARIABLES)].head())

DataFrame enriquecido: 127102 filas x 26 columnas


,ANIO,MES,DIA_SEMANA,ES_FIN_DE_SEMANA,FRANJA_HORARIA,ESTADO_INTERSECCION,TOTAL_VEHICULOS_INVOLUCRADOS,ES_GRAVE
0,2023,9,Martes,0,Mañana,Sin intersección registrada,2,0
1,2023,9,Miércoles,0,Tarde,Con intersección registrada,2,0
2,2023,9,Martes,0,Madrugada,Sin intersección registrada,2,0
3,2023,9,Miércoles,0,Tarde,Con intersección registrada,2,0
4,2023,9,Lunes,0,Tarde,Sin intersección registrada,2,0


### 4.1 Variables derivadas: fórmula o regla, significado y utilidad

In [8]:
pd.set_option("display.max_colwidth", None)
display(enriquecimiento.catalogo_variables())

,variable,formula_o_regla,significado,utilidad
0,ANIO,CRASH_DATE.year,Año calendario del accidente (2019–2023).,Permite revisar si la distribución de severidad cambia entre años y si un patrón es estable en el tiempo.
1,MES,CRASH_DATE.month (1 = enero … 12 = diciembre),Mes del año en que ocurrió el accidente.,"Detecta estacionalidad en la severidad (por ejemplo, meses con más accidentes graves) para planear campañas de prevención."
2,DIA_SEMANA,Nombre del día según CRASH_DATE.dayofweek (Lunes … Domingo),Día de la semana del accidente.,Identifica los días con mayor concentración de accidentes graves para orientar el control en vía.
3,ES_FIN_DE_SEMANA,1 si DIA_SEMANA es Sábado o Domingo; 0 en otro caso,Indicador de accidente ocurrido en fin de semana.,Resume en un solo indicador la comparación entre días laborales y fin de semana.
4,FRANJA_HORARIA,"hora = CRASH_TIME // 100; Madrugada 00–05, Mañana 06–11, Tarde 12–17, Noche 18–23; 'Sin dato' si CRASH_TIME es nulo",Tramo del día en que ocurrió el accidente.,Responde directamente la pregunta por franja horaria y focaliza los horarios de control y prevención.
5,ESTADO_INTERSECCION,'Con intersección registrada' si INTERSECTION_NO o INTERSECTION_DESC no son nulos; 'Sin intersección registrada' en otro caso,Si el accidente tiene una intersección asociada en el registro.,Responde la pregunta por presencia de intersección. Se contrasta con ACCIDENT_TYPE; 'sin registro' no prueba por sí solo que no exista.
6,TOTAL_VEHICULOS_INVOLUCRADOS,TOTAL_BIKE + TOTAL_TRUCK + TOTAL_HEAVY_TRUCK + TOTAL_MOTOR_CYCLE + TOTAL_OTHER_VEHICLES (sin peatones); nulo si algún sumando es nulo,Cantidad de vehículos que participaron en el accidente.,Mide la complejidad del evento y permite ver si más vehículos se asocian a mayor severidad.
7,ES_GRAVE,1 si SEVERITY es Fatal u Hospital; 0 en otro caso,Indicador de accidente con víctimas fatales o que requirieron hospital.,"Permite calcular una tasa de gravedad comparable entre categorías de calzada, intersección, franja y vehículos."


### 4.2 Preguntas adicionales derivadas de la necesidad

Cada pregunta se responde con una tabla de la sección 5 y se apoya en las variables derivadas. Las cuatro primeras corresponden a la pregunta principal (calzada, intersección, franja horaria y tipo de vehículo).

In [9]:
display(enriquecimiento.preguntas_adicionales())

,id,pregunta,variables,tabla
0,P1,¿Cómo se distribuye la severidad según el tipo de calzada (CWAY)?,ES_GRAVE,tabla_calzada
1,P2,¿Cambia la severidad entre accidentes con y sin intersección registrada?,"ESTADO_INTERSECCION, ES_GRAVE",tabla_interseccion
2,P3,¿En qué franja horaria se concentran los accidentes graves?,"FRANJA_HORARIA, ES_GRAVE",tabla_franja_horaria
3,P4,¿Qué tipos de vehículos participan en accidentes y con qué severidad?,ES_GRAVE,tabla_tipo_vehiculo
4,P5,¿La proporción de accidentes graves cambió entre 2019 y 2023?,"ANIO, ES_GRAVE",tabla_anio
5,P6,¿Hay meses con mayor proporción de accidentes graves?,"MES, ES_GRAVE",tabla_mes
6,P7,¿Son más graves los accidentes en fin de semana que en días laborales?,"DIA_SEMANA, ES_FIN_DE_SEMANA, ES_GRAVE",tabla_dia_semana
7,P8,¿Aumenta la gravedad cuando participan más vehículos?,"TOTAL_VEHICULOS_INVOLUCRADOS, ES_GRAVE",tabla_total_vehiculos


### 4.3 Sustento de `ESTADO_INTERSECCION`

Los nulos de intersección no se imputan. Para saber si "sin intersección registrada" es una señal confiable, se contrasta con `ACCIDENT_TYPE`, que describe el tipo de accidente de forma independiente.

In [10]:
contraste_interseccion = enriquecimiento.contrastar_interseccion(df_enriquecido)
display(contraste_interseccion)

,ESTADO_INTERSECCION,Intersection,Midblock,Roads Open To Public Access,Total
0,Con intersección registrada,68109,14,0,68123
1,Sin intersección registrada,4,58974,1,58979
2,Total,68113,58988,1,127102


**Lectura:** si los registros "Con intersección registrada" coinciden casi por completo con `ACCIDENT_TYPE = Intersection`, la regla es consistente. Las diferencias que aparezcan en la tabla son los casos que no coinciden y se mantienen sin corregir. Aun así, la variable describe el **registro**, no una verificación en terreno.

## 5. Tablas de resultados por pregunta

Cada tabla muestra conteos (`n`) y porcentajes. `% del total de accidentes` es la participación de la fila en todos los accidentes; `% de la fila` es la distribución de severidad dentro de esa fila. **Graves** = Fatal + Hospital (`ES_GRAVE = 1`).

In [11]:
tablas = enriquecimiento.construir_tablas(df_enriquecido)
resumen_verificacion = enriquecimiento.verificar_tablas(df_enriquecido, tablas)
display(resumen_verificacion)

,tabla,filas,total_accidentes,excluyente,estado
0,tabla_calzada,4,127102,Sí,OK
1,tabla_interseccion,3,127102,Sí,OK
2,tabla_franja_horaria,6,127102,Sí,OK
3,tabla_tipo_vehiculo,6,127102,No (un accidente puede estar en varias filas),OK
4,tabla_anio,6,127102,Sí,OK
5,tabla_mes,13,127102,Sí,OK
6,tabla_dia_semana,8,127102,Sí,OK
7,tabla_total_vehiculos,5,127102,Sí,OK


### 5.1 Respuesta a la pregunta principal

#### Pregunta 1. Severidad según el tipo de calzada (`CWAY`)

In [12]:
display(tablas["tabla_calzada"])
display(Markdown("**Interpretación:** " + enriquecimiento.interpretar_tabla("tabla_calzada", tablas["tabla_calzada"])))

,Tipo de calzada (CWAY),Accidentes (n),% del total de accidentes,Fatal (n),Hospital (n),Medical (n),PDO Major (n),PDO Minor (n),Fatal (% de la fila),Hospital (% de la fila),Medical (% de la fila),PDO Major (% de la fila),PDO Minor (% de la fila),Graves (n),Graves (% de la fila)
0,L,44758,35.21,96,1645,6844,24205,11968,0.21,3.68,15.29,54.08,26.74,1741,3.89
1,R,12390,9.75,41,483,1721,6769,3376,0.33,3.90,13.89,54.63,27.25,524,4.23
2,S,69954,55.04,598,4136,7920,37894,19406,0.85,5.91,11.32,54.17,27.74,4734,6.77
3,Total,127102,100.00,735,6264,16485,68868,34750,0.58,4.93,12.97,54.18,27.34,6999,5.51


**Interpretación:** Tasa global de accidentes graves (Fatal + Hospital): 5.51 %. Mayor volumen: S con 69,954 accidentes (55.04 % del total). Mayor proporción de graves: S con 6.77 %; la menor: L con 3.89 % (categorías con al menos 100 accidentes). Limitaciones: No hay datos de tránsito ni de longitud de cada tipo de calzada; un mayor número de accidentes puede reflejar más exposición y no más riesgo.

#### Pregunta 2. Severidad según la presencia de intersección

In [13]:
display(tablas["tabla_interseccion"])
display(Markdown("**Interpretación:** " + enriquecimiento.interpretar_tabla("tabla_interseccion", tablas["tabla_interseccion"])))

,Estado de intersección,Accidentes (n),% del total de accidentes,Fatal (n),Hospital (n),Medical (n),PDO Major (n),PDO Minor (n),Fatal (% de la fila),Hospital (% de la fila),Medical (% de la fila),PDO Major (% de la fila),PDO Minor (% de la fila),Graves (n),Graves (% de la fila)
0,Con intersección registrada,68123,53.6,154,2733,10490,37277,17469,0.23,4.01,15.40,54.72,25.64,2887,4.24
1,Sin intersección registrada,58979,46.4,581,3531,5995,31591,17281,0.99,5.99,10.16,53.56,29.30,4112,6.97
2,Total,127102,100.0,735,6264,16485,68868,34750,0.58,4.93,12.97,54.18,27.34,6999,5.51


**Interpretación:** Tasa global de accidentes graves (Fatal + Hospital): 5.51 %. Mayor volumen: Con intersección registrada con 68,123 accidentes (53.60 % del total). Mayor proporción de graves: Sin intersección registrada con 6.97 %; la menor: Con intersección registrada con 4.24 % (categorías con al menos 100 accidentes). Limitaciones: 'Sin intersección registrada' significa que el dato falta; se contrasta con ACCIDENT_TYPE, pero no se verifica en terreno.

#### Pregunta 3. Severidad según la franja horaria

In [14]:
display(tablas["tabla_franja_horaria"])
display(Markdown("**Interpretación:** " + enriquecimiento.interpretar_tabla("tabla_franja_horaria", tablas["tabla_franja_horaria"])))

,Franja horaria,Accidentes (n),% del total de accidentes,Fatal (n),Hospital (n),Medical (n),PDO Major (n),PDO Minor (n),Fatal (% de la fila),Hospital (% de la fila),Medical (% de la fila),PDO Major (% de la fila),PDO Minor (% de la fila),Graves (n),Graves (% de la fila)
0,Madrugada,5355,4.21,106,492,486,3023,1248,1.98,9.19,9.08,56.45,23.31,598,11.17
1,Mañana,41208,32.42,205,1763,5810,21895,11535,0.50,4.28,14.10,53.13,27.99,1968,4.78
2,Tarde,60278,47.42,242,2580,7969,32910,16577,0.40,4.28,13.22,54.60,27.50,2822,4.68
3,Noche,20152,15.85,180,1415,2165,11033,5359,0.89,7.02,10.74,54.75,26.59,1595,7.91
4,Sin dato,109,0.09,2,14,55,7,31,1.83,12.84,50.46,6.42,28.44,16,14.68
5,Total,127102,100.00,735,6264,16485,68868,34750,0.58,4.93,12.97,54.18,27.34,6999,5.51


**Interpretación:** Tasa global de accidentes graves (Fatal + Hospital): 5.51 %. Mayor volumen: Tarde con 60,278 accidentes (47.42 % del total). Mayor proporción de graves: Madrugada con 11.17 %; la menor: Tarde con 4.68 % (categorías con al menos 100 accidentes). Limitaciones: Las franjas tienen distinta duración efectiva de tráfico; sin volumen horario no se puede hablar de riesgo, solo de registros. Las horas nulas se agrupan en 'Sin dato'.

#### Pregunta 4. Severidad según el tipo de vehículo involucrado

Un accidente puede involucrar varios tipos de vehículo, por lo que las filas **no son excluyentes** y los porcentajes del total no suman 100 %.

In [15]:
display(tablas["tabla_tipo_vehiculo"])
display(Markdown("**Interpretación:** " + enriquecimiento.interpretar_tabla("tabla_tipo_vehiculo", tablas["tabla_tipo_vehiculo"])))

,Tipo de vehículo,Accidentes (n),% del total de accidentes,Fatal (n),Hospital (n),Medical (n),PDO Major (n),PDO Minor (n),Fatal (% de la fila),Hospital (% de la fila),Medical (% de la fila),PDO Major (% de la fila),PDO Minor (% de la fila),Graves (n),Graves (% de la fila)
0,Bicicleta,2539,2.00,14,386,592,187,1360,0.55,15.20,23.32,7.37,53.56,400,15.75
1,Camión,4530,3.56,43,198,489,2625,1175,0.95,4.37,10.79,57.95,25.94,241,5.32
2,Camión pesado,2318,1.82,52,144,204,1375,543,2.24,6.21,8.80,59.32,23.43,196,8.46
3,Motocicleta,4334,3.41,158,1286,578,1014,1298,3.65,29.67,13.34,23.40,29.95,1444,33.32
4,Otros vehículos,118208,93.00,603,4993,15121,65124,32367,0.51,4.22,12.79,55.09,27.38,5596,4.73
5,Todos los accidentes (referencia),127102,100.00,735,6264,16485,68868,34750,0.58,4.93,12.97,54.18,27.34,6999,5.51


**Interpretación:** Tasa global de accidentes graves (Fatal + Hospital): 5.51 %. Mayor volumen: Otros vehículos con 118,208 accidentes (93.00 % del total). Mayor proporción de graves: Motocicleta con 33.32 %; la menor: Otros vehículos con 4.73 % (categorías con al menos 100 accidentes). Limitaciones: Las filas no son excluyentes (un accidente puede involucrar varios tipos), por eso los porcentajes del total no suman 100. 'Otros vehículos' agrupa tipos no detallados en el dataset.

### 5.2 Preguntas adicionales

#### Pregunta 5. Severidad por año (`ANIO`)

In [16]:
display(tablas["tabla_anio"])
display(Markdown("**Interpretación:** " + enriquecimiento.interpretar_tabla("tabla_anio", tablas["tabla_anio"])))

,Año,Accidentes (n),% del total de accidentes,Fatal (n),Hospital (n),Medical (n),PDO Major (n),PDO Minor (n),Fatal (% de la fila),Hospital (% de la fila),Medical (% de la fila),PDO Major (% de la fila),PDO Minor (% de la fila),Graves (n),Graves (% de la fila)
0,2019,26685,20.99,150,1334,3115,14523,7563,0.56,5.00,11.67,54.42,28.34,1484,5.56
1,2020,22975,18.08,131,1287,3318,11928,6311,0.57,5.60,14.44,51.92,27.47,1418,6.17
2,2021,26566,20.90,149,1229,3380,14406,7402,0.56,4.63,12.72,54.23,27.86,1378,5.19
3,2022,25057,19.71,158,1247,3323,13619,6710,0.63,4.98,13.26,54.35,26.78,1405,5.61
4,2023,25819,20.31,147,1167,3349,14392,6764,0.57,4.52,12.97,55.74,26.20,1314,5.09
5,Total,127102,100.00,735,6264,16485,68868,34750,0.58,4.93,12.97,54.18,27.34,6999,5.51


**Interpretación:** Tasa global de accidentes graves (Fatal + Hospital): 5.51 %. Mayor volumen: 2019 con 26,685 accidentes (20.99 % del total). Mayor proporción de graves: 2020 con 6.17 %; la menor: 2023 con 5.09 % (categorías con al menos 100 accidentes). Limitaciones: El periodo incluye años afectados por cambios de movilidad; la comparación entre años es descriptiva.

#### Pregunta 6. Severidad por mes (`MES`)

In [17]:
display(tablas["tabla_mes"])
display(Markdown("**Interpretación:** " + enriquecimiento.interpretar_tabla("tabla_mes", tablas["tabla_mes"])))

,Mes,Accidentes (n),% del total de accidentes,Fatal (n),Hospital (n),Medical (n),PDO Major (n),PDO Minor (n),Fatal (% de la fila),Hospital (% de la fila),Medical (% de la fila),PDO Major (% de la fila),PDO Minor (% de la fila),Graves (n),Graves (% de la fila)
0,01 - Enero,9152,7.20,70,493,1208,4812,2569,0.76,5.39,13.20,52.58,28.07,563,6.15
1,02 - Febrero,10363,8.15,46,511,1457,5428,2921,0.44,4.93,14.06,52.38,28.19,557,5.37
2,03 - Marzo,10978,8.64,58,588,1426,5781,3125,0.53,5.36,12.99,52.66,28.47,646,5.88
3,04 - Abril,8802,6.93,57,452,1084,4881,2328,0.65,5.14,12.32,55.45,26.45,509,5.78
4,05 - Mayo,11094,8.73,62,528,1497,5986,3021,0.56,4.76,13.49,53.96,27.23,590,5.32
5,06 - Junio,11128,8.76,60,485,1488,6193,2902,0.54,4.36,13.37,55.65,26.08,545,4.90
6,07 - Julio,11119,8.75,56,534,1510,6041,2978,0.50,4.80,13.58,54.33,26.78,590,5.31
7,08 - Agosto,11557,9.09,65,522,1592,6253,3125,0.56,4.52,13.78,54.11,27.04,587,5.08
8,09 - Septiembre,10467,8.24,65,532,1389,5620,2861,0.62,5.08,13.27,53.69,27.33,597,5.70
9,10 - Octubre,10877,8.56,60,574,1405,5867,2971,0.55,5.28,12.92,53.94,27.31,634,5.83


**Interpretación:** Tasa global de accidentes graves (Fatal + Hospital): 5.51 %. Mayor volumen: 08 - Agosto con 11,557 accidentes (9.09 % del total). Mayor proporción de graves: 01 - Enero con 6.15 %; la menor: 06 - Junio con 4.90 % (categorías con al menos 100 accidentes). Limitaciones: Con cinco años por mes, un solo año atípico puede mover el resultado; no se controla el clima ni el volumen de viajes.

#### Pregunta 7. Severidad por día de la semana (`DIA_SEMANA`, `ES_FIN_DE_SEMANA`)

In [18]:
display(tablas["tabla_dia_semana"])
fin_semana = (
    df_enriquecido.groupby("ES_FIN_DE_SEMANA")["ES_GRAVE"]
    .agg(accidentes="size", graves="sum")
    .assign(pct_graves=lambda t: (100 * t["graves"] / t["accidentes"]).round(2))
    .rename(index={0: "Día laboral", 1: "Fin de semana"})
)
display(fin_semana)
display(Markdown("**Interpretación:** " + enriquecimiento.interpretar_tabla("tabla_dia_semana", tablas["tabla_dia_semana"])))

,Día de la semana,Accidentes (n),% del total de accidentes,Fatal (n),Hospital (n),Medical (n),PDO Major (n),PDO Minor (n),Fatal (% de la fila),Hospital (% de la fila),Medical (% de la fila),PDO Major (% de la fila),PDO Minor (% de la fila),Graves (n),Graves (% de la fila)
0,Lunes,17663,13.90,62,820,2395,9429,4957,0.35,4.64,13.56,53.38,28.06,882,4.99
1,Martes,20213,15.90,99,918,2802,10833,5561,0.49,4.54,13.86,53.59,27.51,1017,5.03
2,Miércoles,19914,15.67,107,873,2687,10886,5361,0.54,4.38,13.49,54.67,26.92,980,4.92
3,Jueves,20689,16.28,98,899,2760,11163,5769,0.47,4.35,13.34,53.96,27.88,997,4.82
4,Viernes,21956,17.27,117,1052,2724,12071,5992,0.53,4.79,12.41,54.98,27.29,1169,5.32
5,Sábado,15262,12.01,126,889,1776,8429,4042,0.83,5.82,11.64,55.23,26.48,1015,6.65
6,Domingo,11405,8.97,126,813,1341,6057,3068,1.10,7.13,11.76,53.11,26.90,939,8.23
7,Total,127102,100.00,735,6264,16485,68868,34750,0.58,4.93,12.97,54.18,27.34,6999,5.51


,accidentes,graves,pct_graves
ES_FIN_DE_SEMANA,,,
Día laboral,100435,5045,5.02
Fin de semana,26667,1954,7.33


**Interpretación:** Tasa global de accidentes graves (Fatal + Hospital): 5.51 %. Mayor volumen: Viernes con 21,956 accidentes (17.27 % del total). Mayor proporción de graves: Domingo con 8.23 %; la menor: Jueves con 4.82 % (categorías con al menos 100 accidentes). Limitaciones: Los fines de semana tienen otro patrón de viajes; sin exposición no se puede afirmar que sean más peligrosos.

#### Pregunta 8. Severidad según el total de vehículos involucrados

In [19]:
display(tablas["tabla_total_vehiculos"])
display(Markdown("**Interpretación:** " + enriquecimiento.interpretar_tabla("tabla_total_vehiculos", tablas["tabla_total_vehiculos"])))

,Vehículos involucrados,Accidentes (n),% del total de accidentes,Fatal (n),Hospital (n),Medical (n),PDO Major (n),PDO Minor (n),Fatal (% de la fila),Hospital (% de la fila),Medical (% de la fila),PDO Major (% de la fila),PDO Minor (% de la fila),Graves (n),Graves (% de la fila)
0,0 vehículos,5292,4.16,4,377,964,2655,1292,0.08,7.12,18.22,50.17,24.41,381,7.20
1,1 vehículo,31178,24.53,491,2996,3064,15296,9331,1.57,9.61,9.83,49.06,29.93,3487,11.18
2,2 vehículos,84111,66.18,209,2538,10652,47052,23660,0.25,3.02,12.66,55.94,28.13,2747,3.27
3,3 o más vehículos,6521,5.13,31,353,1805,3865,467,0.48,5.41,27.68,59.27,7.16,384,5.89
4,Total,127102,100.00,735,6264,16485,68868,34750,0.58,4.93,12.97,54.18,27.34,6999,5.51


**Interpretación:** Tasa global de accidentes graves (Fatal + Hospital): 5.51 %. Mayor volumen: 2 vehículos con 84,111 accidentes (66.18 % del total). Mayor proporción de graves: 1 vehículo con 11.18 %; la menor: 2 vehículos con 3.27 % (categorías con al menos 100 accidentes). Limitaciones: Suma bicicletas, camiones, camiones pesados, motos y otros vehículos; no incluye peatones. Los accidentes con 0 vehículos registrados pueden ser casos con solo peatones o datos incompletos.

### 5.3 Verificación independiente con Nemotron Parse

Cada tabla (en bloques de hasta 6 filas, porque las tablas altas no se leen bien) se dibuja como imagen y `nvidia/nemotron-parse` la lee; se compara qué porcentaje de los conteos reales aparece en la lectura del modelo. Es un control de lectura: **no genera ni modifica cifras**. Si no hay `API_KEY_NVIDIA` o la API no responde, la tabla queda como "Omitida" y el flujo continúa.

In [20]:
cliente_nvidia = ClienteNvidia(timeout=60)
validacion_llm = enriquecimiento.validar_tablas_con_llm(tablas, cliente_nvidia)
display(validacion_llm)

,tabla,cifras_esperadas,cifras_leidas,coincidencia_pct,estado
0,tabla_calzada,28,29,100.0,Coincide (modelo: nvidia/nemotron-parse (img))
1,tabla_interseccion,21,22,100.0,Coincide (modelo: nvidia/nemotron-parse (img))
2,tabla_franja_horaria,40,43,100.0,Coincide (modelo: nvidia/nemotron-parse (img))
3,tabla_tipo_vehiculo,42,46,100.0,Coincide (modelo: nvidia/nemotron-parse (img))
4,tabla_anio,42,49,100.0,Coincide (modelo: nvidia/nemotron-parse (img))
5,tabla_mes,87,101,100.0,Coincide (modelo: nvidia/nemotron-parse (img))
6,tabla_dia_semana,55,58,100.0,Coincide (modelo: nvidia/nemotron-parse (img))
7,tabla_total_vehiculos,34,39,100.0,Coincide (modelo: nvidia/nemotron-parse (img))


## 5.4 Exportación y verificación de consistencia

Se exportan a `AE3/resultados` el dataset enriquecido y, por cada tabla, un `.csv` (para la visualización) y un `.txt`. Después se vuelven a leer los archivos para comprobar que coinciden con lo mostrado en este notebook.

In [21]:
archivos_exportados = enriquecimiento.exportar_resultados(df_enriquecido, tablas, RESULTADOS_DIR)
validacion_llm.to_csv(RESULTADOS_DIR / "validacion_llm_tablas.csv", index=False)
verificacion_exportacion = enriquecimiento.verificar_exportacion(df_enriquecido, tablas, RESULTADOS_DIR)
display(verificacion_exportacion)
print(f"{len(archivos_exportados)} archivos exportados en {RESULTADOS_DIR.relative_to(RAIZ_REPO)}")

,archivo,estado,filas
0,WA_Crash_enriquecido.csv,OK,127102
1,tabla_calzada.csv/.txt,OK,4
2,tabla_interseccion.csv/.txt,OK,3
3,tabla_franja_horaria.csv/.txt,OK,6
4,tabla_tipo_vehiculo.csv/.txt,OK,6
5,tabla_anio.csv/.txt,OK,6
6,tabla_mes.csv/.txt,OK,13
7,tabla_dia_semana.csv/.txt,OK,8
8,tabla_total_vehiculos.csv/.txt,OK,5


17 archivos exportados en AE3/resultados


## 6. Resumen descriptivo del DataFrame final

`df_final` es el DataFrame **transformado y enriquecido**. Los reportes se regeneran en cada ejecución y se comprueba que coincidan con el DataFrame de esa ejecución.


In [22]:
df_final = df_enriquecido
reportes = Reportes()
ruta_resumen = RESULTADOS_DIR / "resumen_dataframe.txt"
ruta_metadata = RESULTADOS_DIR / "resumen_dataframe_metadata.txt"

texto_resumen = reportes.generar_reporte(
    df_final,
    dataset_id="kalley/WA_Crash",
    split="train",
    es_muestra=False,
)
estadisticas = df_final.describe(include="all").to_string()
texto_resumen += "\nEstadísticas descriptivas (todas las columnas):\n" + estadisticas + "\n"
ruta_resumen.write_text(texto_resumen, encoding="utf-8")

metadata_final = reportes.reporte_descriptivo(df_final)
reportes.guardar_reporte_metadata(metadata_final, ruta_metadata)

contenido_resumen = ruta_resumen.read_text(encoding="utf-8")
contenido_metadata = ruta_metadata.read_text(encoding="utf-8")
assert f"Filas: {len(df_final)}" in contenido_resumen
assert f"Columnas: {len(df_final.columns)}" in contenido_resumen
seccion_tipos = contenido_resumen.split("Columnas y tipos:\n", 1)[1].split("\n\nValores nulos por columna:", 1)[0]
seccion_nulos = contenido_resumen.split("Valores nulos por columna:\n", 1)[1].split("\n\nEstadísticas numéricas:", 1)[0]
assert all(f"- {columna}: {df_final[columna].dtype}" in seccion_tipos for columna in df_final.columns)
assert all(f"- {columna}: {int(df_final[columna].isna().sum())}" in seccion_nulos for columna in df_final.columns)
assert contenido_metadata == metadata_final.to_string(index=False) + "\n"
assert len(metadata_final) == len(df_final.columns)
assert ruta_resumen.stat().st_size > 0 and ruta_metadata.stat().st_size > 0
print(f"Reportes verificados: {ruta_resumen.relative_to(RAIZ_REPO)} y {ruta_metadata.relative_to(RAIZ_REPO)}")

Reportes verificados: AE3/resultados/resumen_dataframe.txt y AE3/resultados/resumen_dataframe_metadata.txt
